In [4]:
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()

base_time = datetime.now()
base_date = datetime(2026, 4, 20)
sql = []

# =========================
# CONFIG
# =========================
stations = 100
drivers = 500
vehicles = 200
routes = 300
schedules = 1000
passengers = 3000
trips = 2000
tickets = 4000
boarding_count = 5000

start_datetime = datetime(2026, 3, 1, 5, 0, 0)

# =========================
# 1. STATION
# =========================
zones = ['North', 'South', 'East', 'West', 'Central', None, '']

for i in range(1, stations + 1):
    zone = random.choice(zones)

    # duplicate some names intentionally
    name = f"Station {random.randint(1, 70)}"

    if zone is None:
        zone_value = 'NULL'
    else:
        zone_value = f"'{zone}'"

    sql.append(f"INSERT INTO Station VALUES ({i}, {zone_value}, '{name}');")

# =========================
# 2. DRIVER
# =========================
for i in range(1, drivers + 1):
    name = fake.name().replace("'", "''")
    sql.append(f"INSERT INTO Driver VALUES ({i}, '{name}', 'LIC{100000+i}');")

# =========================
# 3. VEHICLE
# =========================
for i in range(1, vehicles + 1):
    vtype = random.choice(["Bus", "Minibus", "Van"])
    sql.append(f"INSERT INTO Vehicle VALUES ({i}, {random.randint(20,80)}, '{vtype}');")

# =========================
# 4. ROUTE
# =========================
for i in range(1, routes + 1):
    sql.append(f"INSERT INTO Route VALUES ({i}, 'Route {i}', 'Start {i%50}', 'End {i%50}');")

# =========================
# 5. SCHEDULE
# =========================
for i in range(1, schedules + 1):
    dep = start_datetime + timedelta(
    days=random.randint(0, 60),
    hours=random.randint(5, 23),
    minutes=random.randint(0, 59)
)
    arr = dep + timedelta(minutes=30)
    sql.append(f"INSERT INTO Schedule VALUES ({i}, '{arr}', '{dep}');")

# =========================
# 6. PASSENGER
# =========================
for i in range(1, passengers + 1):
    name = fake.name().replace("'", "''")
    phone = fake.phone_number().replace("'", "''")
    email = fake.email().replace("'", "''")
    sql.append(f"INSERT INTO Passenger VALUES ({i}, '{name}', '{phone}', '{email}');")

# =========================
# 7. TRIP
# =========================
for i in range(1, trips + 1):

    schedule_id = random.randint(1, schedules)
    route_id = random.randint(1, routes)
    vehicle_id = random.randint(1, vehicles)

    actual_departure = base_time + timedelta(minutes=i * 5)
    actual_arrival = actual_departure + timedelta(minutes=random.randint(20, 60))
    trip_date = base_date + timedelta(days=random.randint(0, 7))

    sql.append(f"""INSERT INTO Trip VALUES (
        {i},
        '{random.choice(['Scheduled','Completed','Cancelled'])}',
        '{trip_date.date()}',
        '{actual_departure}',
        '{actual_arrival}',
        {random.randint(0,50)},
        {schedule_id},
        {route_id},
        {vehicle_id}
    );""")

# =========================
# 8. TICKET
# =========================
ticket_prices = {}

for i in range(1, tickets + 1):
    price = random.randint(5, 50)
    ticket_prices[i] = price

    sql.append(f"INSERT INTO Ticket VALUES ({i}, {price}, '{random.choice(
        ['Standard','VIP','Student','Premium']
    )}', {random.randint(1, passengers)}, {random.randint(1, trips)});")

# =========================
# 9. PAYMENT
# =========================
base_payment_time = datetime(2026, 3, 1, 6, 0, 0)

for i in range(1, tickets + 1):
    pay_time = base_payment_time + timedelta(
        days=random.randint(0, 60),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )

    price = ticket_prices[i]
    

    sql.append(f"""INSERT INTO Payment VALUES (
        {i},
        {price},
        '{random.choice(['Card','Cash','Online'])}',
        '{pay_time}',
        {i}
    );""")
    
# =========================
# 10. BOARDING
# =========================

trip_usage = {i: 0 for i in range(1, trips + 1)}


trip_capacity = {i: random.randint(20, 80) for i in range(1, trips + 1)}

boarding_id = 1

for _ in range(boarding_count * 2):  

    trip_id = random.randint(1, trips)

   
    if trip_usage[trip_id] >= trip_capacity[trip_id]:
        continue

    trip_usage[trip_id] += 1

    sql.append(f"""INSERT INTO Boarding VALUES (
        {boarding_id},
        '{start_datetime + timedelta(
            days=random.randint(0, 60),
            hours=random.randint(5, 23),
            minutes=random.randint(0, 59)
        )}',
        '{random.choice(['Boarded','Missed','Pending'])}',
        'S{random.randint(1,60)}',
        {str(random.choice([True, False])).upper()},
        'P{random.randint(1,10)}',
        {random.randint(1, tickets)},
        {random.randint(1, stations)},
        {trip_id}
    );""")

    boarding_id += 1

    if boarding_id > boarding_count:
        break

# =========================
# 11. ROUTE_STOP
# =========================
for route_id in range(1, routes + 1):

    if random.random() < 0.2:
        continue

    num_stops = random.randint(5, 15)
    stations_list = random.sample(range(1, stations + 1), num_stops)

    base_time = start_datetime + timedelta(days=random.randint(0, 60))

    for stop_number, station_id in enumerate(stations_list, start=1):

        arrival = base_time + timedelta(minutes=stop_number * random.randint(5, 20))
        departure = arrival + timedelta(minutes=random.randint(2, 10))

        sql.append(f"""INSERT INTO Route_stop VALUES (
            {route_id},
            {stop_number},
            {station_id},
            '{arrival}',
            '{departure}'
        );""")

# =========================
# 12. ASSIGNMENT
# =========================

now = datetime.now()

for i in range(1, 2000 + 1):

    # assignments spread: some past, some ongoing, some future
    start_offset_hours = random.randint(-72, 72)
    duration_hours = random.randint(1, 5)

    start_time = now + timedelta(hours=start_offset_hours)
    end_time = start_time + timedelta(hours=duration_hours)

    sql.append(f"""INSERT INTO Assignment VALUES (
        {i},
        '{start_time}',
        '{end_time}',
        {random.randint(1, drivers)},
        {random.randint(1, vehicles)},
        {random.randint(1, trips)}
    );""")


with open("dataset.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(sql))

print("dataset.sql generated successfully!")

dataset.sql generated successfully!
